# Experimento 7 — Iterative Magnitude Pruning

**Pergunta.** Ao podar iterativamente 20% dos pesos restantes e rebobinar para init, a rede perde primeiro a capacidade de usar forma ou cor?

**Resultado até 89,3%.** Todas as 30 trajetórias adquiriram cor na época 1 e depois escaparam. Forma dominou a decisão final (`shape sensitivity ≈0,985–0,991`; color sensitivity ≈0,009–0,014 nas médias). A poda intermediária frequentemente antecipou o escape; em 89,3% apareceu pequena deterioração, mas nenhuma seed ficou presa. Rodadas 11–13 estendem o teste a 91,4%, 93,1% e 94,5%. Sensibilidade mede efeito causal na saída, não decodificabilidade interna.

In [ ]:
from pathlib import Path
import os,subprocess,sys,pandas as pd,matplotlib.pyplot as plt
ROOT=Path.cwd().resolve(); ROOT=ROOT.parent if ROOT.name=='notebooks' else ROOT
if not (ROOT/'src').exists() and Path('/content/src').exists(): ROOT=Path('/content')
os.chdir(ROOT); sys.path.insert(0,str(ROOT))
RUN_EXPERIMENT=False
IMP_OUTPUT=Path('outputs/sanity_check/imp')
# Para Drive remoto, use Path('/content/drive/MyDrive/pruning-shortcuts/imp').
if RUN_EXPERIMENT:
    subprocess.run([sys.executable,'scripts/run_imp.py','--config','configs/sanity_check.yaml','--rounds','13','--output-dir',str(IMP_OUTPUT)],check=True)

In [ ]:
path=IMP_OUTPUT/'round_summary.csv'
assert path.exists(), f'IMP summary not found: {path}'
summary=pd.read_csv(path); summary['escaped']=summary.escape_epoch.notna()
display(summary.groupby(['round','sparsity']).agg(seeds=('seed','count'),escaped=('escaped','sum'),escape_epoch=('escape_epoch','mean'),shortcut_auc=('shortcut_auc','mean'),conflicting=('final_conflicting_accuracy','mean'),worst_group=('final_worst_group_accuracy','mean'),color_sensitivity=('final_color_swap_sensitivity','mean'),shape_sensitivity=('final_shape_swap_sensitivity','mean')).reset_index())
for seed,rows in summary.groupby('seed'):
    plt.plot(rows.sparsity,rows.escape_epoch,marker='o',label=f'seed {seed}')
plt.axvline(.9,color='black',ls='--'); plt.xlabel('Sparsity'); plt.ylabel('Escape epoch'); plt.legend(); plt.grid(alpha=.2); plt.show()